<a href="https://colab.research.google.com/github/manojsai7/Stock_SenseAi/blob/master/notebooks/StockSense_Colab_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# StockSense AI: guarded Colab training workflow

This notebook reuses the repository's canonical `stationary-return-v1` training and artifact smoke-test code. It does not train, refresh market data, or export files by default. Set the flags in the next cell deliberately.

**Do not promote a candidate model based on the training validation metrics alone.** This workflow checks that artifacts reload and run; it does not replace a separate untouched walk-forward performance evaluation. Candidate outputs are written outside `models_storage_stationary/` so they cannot silently overwrite the current packages.

Colab is optional for running the existing inference service. Use it for a reproducible training session or if local compute/dependencies are inconvenient—not because training has not happened.

In [2]:
# Explicit controls. The safe defaults perform no data refresh or training.
REFRESH_DATA = False
RUN_TRAINING = False
EXPORT_CANDIDATES_TO_DRIVE = False
SYMBOLS = ['TCS.NS', 'RELIANCE.NS', 'INFY.NS', 'HDFCBANK.NS', 'ICICIBANK.NS']
REPOSITORY_URL = 'https://github.com/manojsai7/Stock_SenseAi.git'

if RUN_TRAINING and not REFRESH_DATA:
    print('Training uses the repository snapshot. Check the displayed source dates before proceeding.')
print({'refresh_data': REFRESH_DATA, 'run_training': RUN_TRAINING, 'export_to_drive': EXPORT_CANDIDATES_TO_DRIVE, 'symbols': SYMBOLS})

{'refresh_data': False, 'run_training': False, 'export_to_drive': False, 'symbols': ['TCS.NS', 'RELIANCE.NS', 'INFY.NS', 'HDFCBANK.NS', 'ICICIBANK.NS']}


In [3]:
from pathlib import Path
import os, subprocess, sys

REPO_ROOT = Path('/content/Stock_SenseAi')
if not (REPO_ROOT / 'scripts' / 'train_production_stationary_model.py').exists():
    subprocess.check_call(['git', 'clone', REPOSITORY_URL, str(REPO_ROOT)])
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
CANDIDATE_ROOT = REPO_ROOT / 'models_storage_stationary_candidate'
CANDIDATE_DATA_ROOT = REPO_ROOT / 'data' / 'candidate_refresh'
print('Repository:', REPO_ROOT)
print('Candidate artifacts:', CANDIDATE_ROOT)

Repository: /content/Stock_SenseAi
Candidate artifacts: /content/Stock_SenseAi/models_storage_stationary_candidate


In [4]:
from pathlib import Path
import pandas as pd

data_files = {}
if REFRESH_DATA:
    from ml.data.acquisition import ResearchDataAcquisition
    acquisition = ResearchDataAcquisition(
        raw_dir=CANDIDATE_DATA_ROOT / 'raw',
        manifests_dir=CANDIDATE_DATA_ROOT / 'manifests',
        processed_dir=CANDIDATE_DATA_ROOT / 'processed',
    )
    for symbol in SYMBOLS:
        result = acquisition.acquire_stock(symbol, force=True)
        if not result.get('is_valid'):
            raise RuntimeError(f'Data validation failed for {symbol}: {result}')
        clean = symbol.replace('.', '_')
        data_files[symbol] = CANDIDATE_DATA_ROOT / 'processed' / f'{clean}_processed.csv'
else:
    for symbol in SYMBOLS:
        clean = symbol.replace('.', '_')
        processed = REPO_ROOT / 'data' / 'processed' / f'{clean}_processed.csv'
        raw = REPO_ROOT / 'data' / 'raw' / f'{clean}.csv'
        data_files[symbol] = processed if processed.exists() else raw

rows = []
for symbol, path in data_files.items():
    if not Path(path).exists():
        raise FileNotFoundError(f'No source data for {symbol}: {path}')
    bars = pd.read_csv(path, usecols=['Date'])
    rows.append({'symbol': symbol, 'source_file': str(path), 'bars': len(bars), 'first_date': bars['Date'].min(), 'last_date': bars['Date'].max()})
pd.DataFrame(rows)

,symbol,source_file,bars,first_date,last_date
0,TCS.NS,/content/Stock_SenseAi/data/processed/TCS_NS_p...,1241,2021-10-07,2026-10-07
1,RELIANCE.NS,/content/Stock_SenseAi/data/processed/RELIANCE...,1241,2021-10-07,2026-10-07
2,INFY.NS,/content/Stock_SenseAi/data/processed/INFY_NS_...,1241,2021-10-07,2026-10-07
3,HDFCBANK.NS,/content/Stock_SenseAi/data/processed/HDFCBANK...,1241,2021-10-07,2026-10-07
4,ICICIBANK.NS,/content/Stock_SenseAi/data/processed/ICICIBAN...,1241,2021-10-07,2026-10-07


## Optional candidate training

Set `RUN_TRAINING = True` only after checking the source dates above. If refreshing data, `REFRESH_DATA` writes a separate candidate snapshot. Each stock is trained into `models_storage_stationary_candidate/`; current production artifacts remain unchanged. The smoke check reloads each candidate package and predicts using the same source file used for that training run.

In [5]:
if RUN_TRAINING:
    import json
    import subprocess

    # Colab may already include TensorFlow; install project dependencies if needed.
    try:
        import tensorflow as tf
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'backend/requirements.txt'])
        import tensorflow as tf

    from scripts.train_production_stationary_model import ProductionStationaryTrainer
    from scripts.run_production_training_all import smoke_test_artifact

    candidate_results = {}
    for symbol in SYMBOLS:
        clean = symbol.replace('.', '_')
        output_dir = CANDIDATE_ROOT / clean
        trainer = ProductionStationaryTrainer(
            symbol=symbol,
            data_file=data_files[symbol],
            output_dir=output_dir,
            val_ratio=0.15,
            min_samples=400,
            max_epochs=40,
            batch_size=32,
            learning_rate=0.001,
            patience=8,
            random_seed=42,
        )
        training = trainer.run()
        smoke = smoke_test_artifact(symbol, output_dir, data_file=Path(data_files[symbol]))
        candidate_results[symbol] = {'training': training, 'smoke_test': smoke}
        print(symbol, 'artifact reload and inference passed:', smoke['predicted_close'])

    summary_path = CANDIDATE_ROOT / 'candidate_training_summary.json'
    summary_path.write_text(json.dumps(candidate_results, indent=2), encoding='utf-8')
    print('Candidate summary:', summary_path)
else:
    print('Training skipped. Set RUN_TRAINING = True in the controls cell to create candidate artifacts.')

Training skipped. Set RUN_TRAINING = True in the controls cell to create candidate artifacts.


In [6]:
if EXPORT_CANDIDATES_TO_DRIVE:
    if not RUN_TRAINING or not (CANDIDATE_ROOT / 'candidate_training_summary.json').exists():
        raise RuntimeError('Train and smoke-check candidates before exporting them.')
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    archive_base = Path('/content/drive/MyDrive/StockSenseAI/candidates') / 'stationary_candidate'
    archive_base.parent.mkdir(parents=True, exist_ok=True)
    archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=CANDIDATE_ROOT)
    print('Exported candidate package archive:', archive_path)
else:
    print('Drive export skipped.')

Drive export skipped.


In [7]:
from pathlib import Path

for p in Path("/content/drive/MyDrive").rglob("*"):
    if p.suffix in [".keras", ".h5", ".pkl", ".json"]:
        print(p)

## Before any production promotion

1. Evaluate candidates with an untouched chronological walk-forward test and compare them with zero-return, training-majority, momentum, and simple statistical baselines.
2. Keep experiment checkpoints and production candidates separate. Never copy one `.keras` file by itself; the model, both scalers, metadata, feature order, and manifest are a single package.
3. Promote only a reviewed candidate package, verify its checksums and inference, then deploy and check the hosted API and website. Training and smoke-test success do not establish predictive skill.

In [8]:
import shutil

shutil.make_archive(
    "/content/StockSense_Colab_Artifacts",
    "zip",
    "/content/drive/MyDrive/PATH_TO_YOUR_MODEL_FOLDER"
)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/PATH_TO_YOUR_MODEL_FOLDER'

In [9]:
from google.colab import files

files.download("/content/StockSense_Colab_Artifacts.zip")

FileNotFoundError: Cannot find file: /content/StockSense_Colab_Artifacts.zip